In [1]:
import os, random
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

random.seed(42)
np.random.seed(42)

ITEMS = {
    "main": ["Burger", "Hotdog", "Chicken Rice", "Sisig Rice", "Spaghetti"],
    "side": ["Fries", "Nuggets", "Lumpia"],
    "drink": ["Iced Tea", "Soda", "Water", "Coffee"],
    "dessert": ["Sundae", "Donut"],
    "addon": ["Extra Rice", "Extra Sauce"]
}

# simple profit proxy (used by promo engine)
MARGINS = {
    "Burger": 35, "Hotdog": 25, "Chicken Rice": 40, "Sisig Rice": 45, "Spaghetti": 30,
    "Fries": 20, "Nuggets": 22, "Lumpia": 18,
    "Iced Tea": 15, "Soda": 12, "Water": 8, "Coffee": 18,
    "Sundae": 20, "Donut": 14,
    "Extra Rice": 10, "Extra Sauce": 6,
    "Cheese Stick": 16  # appears mostly in Dataset B to cause drift
}

def pick_segment():
    return random.choices(["morning","lunch","dinner"], weights=[0.25, 0.45, 0.30])[0]

def pick_day_type():
    return random.choices(["weekday","weekend"], weights=[0.75, 0.25])[0]

def make_basket(profile="A"):
    seg = pick_segment()
    day = pick_day_type()

    # Base (Dataset A: stable)
    p_main = 0.85
    p_drink = 0.65
    p_side  = 0.45
    p_dess  = 0.20
    p_addon = 0.25

    # Dataset B drift: more drinks/desserts + new item appears
    if profile == "B":
        if seg == "lunch":
            p_drink += 0.15
        p_dess += 0.15
        p_side += 0.05

    basket = []

    # Main item
    if random.random() < p_main:
        mains = ITEMS["main"]
        if seg == "morning":
            weights = [0.18, 0.26, 0.16, 0.14, 0.26]
        elif seg == "lunch":
            weights = [0.22, 0.12, 0.28, 0.26, 0.12]
        else:
            weights = [0.26, 0.12, 0.24, 0.30, 0.08]
        basket.append(random.choices(mains, weights=weights)[0])

    # Drink
    if random.random() < p_drink:
        drinks = ITEMS["drink"]
        if seg == "morning":
            weights = [0.12, 0.10, 0.18, 0.60]  # coffee morning
        else:
            weights = [0.45, 0.25, 0.25, 0.05]
        basket.append(random.choices(drinks, weights=weights)[0])

    # Side (correlated with burger/hotdog)
    if random.random() < p_side:
        if "Burger" in basket or "Hotdog" in basket:
            basket.append(random.choices(["Fries","Nuggets","Lumpia"], weights=[0.55,0.30,0.15])[0])
        else:
            basket.append(random.choices(ITEMS["side"], weights=[0.35,0.35,0.30])[0])

    # Dessert
    if random.random() < p_dess:
        basket.append(random.choice(ITEMS["dessert"]))

    # Add-ons (correlated with rice meals)
    if random.random() < p_addon:
        if "Chicken Rice" in basket or "Sisig Rice" in basket:
            basket.append(random.choices(["Extra Rice","Extra Sauce"], weights=[0.65,0.35])[0])
        else:
            basket.append(random.choice(ITEMS["addon"]))

    # Dataset B: new trending item (drift)
    if profile == "B" and random.random() < 0.18:
        basket.append("Cheese Stick")
        if "Coffee" not in basket and random.random() < 0.40:
            basket.append(random.choice(["Iced Tea", "Soda"]))

    # occasional single-item purchase
    if random.random() < 0.10:
        basket = [random.choice(sum(ITEMS.values(), []))]

    return sorted(list(set(basket))), seg, day

def generate_dataset(out_dir, profile, total_tx=1500, batches=3):
    os.makedirs(out_dir, exist_ok=True)
    per_batch = total_tx // batches
    start = datetime(2026, 1, 1)

    # margins file
    pd.DataFrame([{"item":k,"margin_php":v} for k,v in MARGINS.items()]).to_csv(
        os.path.join(out_dir, "margins.csv"), index=False
    )

    tx_id = 1
    for b in range(1, batches+1):
        rows = []
        for _ in range(per_batch):
            basket, seg, day = make_basket(profile)
            ts = start + timedelta(minutes=random.randint(0, 60*24*14))
            rows.append({
                "tx_id": tx_id,
                "timestamp": ts.isoformat(),
                "segment": seg,
                "day_type": day,
                "items": ", ".join(basket)
            })
            tx_id += 1

        pd.DataFrame(rows).to_csv(os.path.join(out_dir, f"batch{b}.csv"), index=False)

if __name__ == "__main__":
    generate_dataset("data/datasetA", "A", total_tx=1500, batches=3)
    generate_dataset("data/datasetB", "B", total_tx=1500, batches=3)
    print("✅ Generated Dataset A & B with 3 batches each.")


✅ Generated Dataset A & B with 3 batches each.
